# Phase 1: one week of inference experiments
Three dedicated VMs (`vllm`, `sglang`, `trtllm`) and one sequential cross-framework VM; one GPU each, 672 GPU-hours total. Use identical pinned model revisions and workloads when comparing frameworks.

Install the repository with `pip install -e '.[notebook]'` in each framework-compatible environment. The notebook runs selected cells interactively; the matching Python script supervises the complete queue. CPU simulation is the default. Set `RUN_REAL_GPU=True` only after filling verified preflight records. Simulation results are not GPU measurements.


In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src" / "llm_optimizer").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from llm_optimizer.experiment.phase1 import Phase1Runner, plot_group


## VM, model, workload, and budget controls
Change `FRAMEWORKS` to a single framework on dedicated VMs, or all three on the comparison VM. All frameworks on one VM run sequentially and must be installed compatibly. Use a separate campaign name for independent replications. Changing any search setting creates a new configuration hash; restarts with identical settings resume existing studies. All campaigns on a VM must share `RUN_ROOT` and the same GPU assignment.

In [ ]:
CAMPAIGN = "phase1-week1"
FRAMEWORKS = ["sglang"]  # or ["vllm", "sglang", "trtllm"]
RUN_REAL_GPU = False
CONCURRENCIES = [1, 8, 32, 128]
SEEDS = [11, 29, 47]
RUN_ROOT = ROOT / "experiment_results" / "phase1"
CAMPAIGN_HOURS = 168
GROUP_TIMEOUT_SECONDS = 6 * 3600  # includes all frameworks/seeds in a group
MAX_RESTARTS = 2  # bounded retries after a crashed/timed-out group
MODEL = "Qwen/Qwen3.5-4B"  # same model across A/B isolates search-space effects
MODEL_SNAPSHOT = "/workspace/models/pinned-model-snapshot"
TRIALS = 40  # attempted trials per framework/concurrency/sampler/seed/pruner
REQUESTS = 896
VALIDATION_REPEATS = 5

# Fill with actual verified values. Never mark checks true without performing them.
PREFLIGHT = {
    framework: {
        "gpu_type": "REPLACE", "framework_version": "REPLACE",
        "cuda_version": "REPLACE", "driver_version": "REPLACE",
        "model_revision": "REPLACE_WITH_MODEL_COMMIT",
        "native_config_verified": False, "quality_verified": False,
        "verified_precision_combinations": [],  # weights/cache pairs approved on this VM
    }
    for framework in ["vllm", "sglang", "trtllm"]
}


## Explicit base and extended search spaces
Every domain below is editable. Base numeric domains must be regularly spaced and contain at least two values. Extended categorical domains must be supported by the installed engine and model. Remove unsupported choices after preflight, including numerical quality checks for cache dtype changes. Graph profiles and padding are conditional. `schedule_conservativeness` is SGLang-only; Phase A retains its fixed baseline value of 1.0, while Phase B searches the explicit domain below.

In [ ]:
PREFILL_BUDGETS = list(range(1024, 8193, 1024))
TOKEN_BUDGETS = list(range(1024, 32769, 256))
CAPACITIES = list(range(8, 129, 8))
BASE_SPACES = {
    1: {
        "vllm": {"max_num_batched_tokens": PREFILL_BUDGETS, "cuda_graph_on": [0, 1]},
        "sglang": {"chunked_prefill_size": PREFILL_BUDGETS, "cuda_graph_on": [0, 1]},
        "trtllm": {"max_num_tokens": PREFILL_BUDGETS, "cuda_graph_on": [0, 1]},
    },
    128: {  # shared regime for all concurrency levels > 1
        "vllm": {
            "max_num_seqs": CAPACITIES, "max_num_batched_tokens": TOKEN_BUDGETS,
            "gpu_memory_utilization": [0.75, 0.80, 0.85, 0.90],
        },
        "sglang": {
            "max_running_requests": CAPACITIES, "chunked_prefill_size": TOKEN_BUDGETS,
            "mem_fraction_static": [0.70, 0.75, 0.80, 0.85],
        },
        "trtllm": {
            "max_batch_size": CAPACITIES, "max_num_tokens": TOKEN_BUDGETS,
            "kv_cache_free_gpu_memory_fraction": [0.65, 0.75, 0.85, 0.95],
        },
    },
}

EXPANDED_CHOICES = {'vllm': {'async_scheduling': [False, True],
          'performance_mode': ['balanced', 'interactivity', 'throughput'],
          'attention_backend': ['auto', 'FLASH_ATTN', 'FLASHINFER'],
          'kv_cache_dtype': ['auto'],  # overridden by PRECISION_CHOICES below
          'enable_prefix_caching': [False, True],
          'graph_mode': ['NONE', 'PIECEWISE', 'FULL_AND_PIECEWISE']},
 'sglang': {'attention_backend': ['triton', 'fa3', 'flashinfer'],
            'disable_radix_cache': [False, True],
            'schedule_policy': ['fcfs', 'lpm'],
            'enable_mixed_chunk': [False, True],
            'disable_overlap_schedule': [False, True],
            'cuda_graph_backend_decode': ['disabled', 'full'],
            'cuda_graph_backend_prefill': ['disabled', 'breakable']},
 'trtllm': {'attn_backend': ['TRTLLM'],
            'enable_chunked_prefill': [False, True],
            'enable_block_reuse': [False, True],
            'capacity_scheduler_policy': ['GUARANTEED_NO_EVICT', 'MAX_UTILIZATION'],
            'graph_enabled': [False, True]}}

EXTRA_SPACES = {
    "vllm": {"graph_capture_profile": ["powers", "dense"]},
    "sglang": {
        "schedule_conservativeness": [0.5, 0.75, 1.0, 1.25, 1.5],
        "max_prefill_tokens": [8192, 16384, 24576, 32768],
    },
    "trtllm": {
        "graph_capture_profile": ["powers", "dense"],
        "graph_padding": [0, 1],
    },
}


## Bind the shared runner
Persistent SQLite storage is local to each VM. Copy artifacts off the VM periodically. A VM-wide file lock prevents overlapping workers under the same root. The runner records abandoned RUNNING trials as FAIL after acquiring exclusive ownership; failed and pruned attempts count toward the configured trial budget. Notebook kernels do not have the batch script’s process-group watchdog. After an interrupted notebook, ensure its server processes have stopped before resuming.

## Weight precision and KV-cache precision
Phase B searches the two axes independently: BF16/BF16, BF16/FP8, FP8/BF16, and FP8/FP8. Set either domain to `["fp8"]` to force FP8. `dtype` remains `bfloat16` for compute/non-quantized layers: FP8 weights use quantization, not `dtype="fp8"`. This uses online/on-load quantization from the same **unquantized pinned source checkpoint**, so models and workloads stay comparable; it does not generate or persist an FP8 weight checkpoint. “Cache” here means the runtime KV cache, not a disk/download cache.

Mappings: vLLM/SGLang use `quantization="fp8"` for FP8 weights (omit for BF16). vLLM uses `kv_cache_dtype="fp8"` plus calculated KV scales; SGLang uses `fp8_e4m3`; BF16 cache uses `auto` with BF16 dtype. TensorRT-LLM uses YAML `quant_config.quant_algo` and `quant_config.kv_cache_quant_algo`, with `FP8` or null, plus calibration settings for FP8 weights. Its KV-cache configuration also explicitly selects `fp8` or the default dtype. Remove incompatible attention backends or other options before launch.

On each GPU VM, verify backend support, calibration/scaling, and output quality for every selected combination and the installed version/model/GPU. Then list the approved pairs in `PREFLIGHT[framework]["verified_precision_combinations"]`, e.g. `["bf16/bf16", "bf16/fp8", "fp8/bf16", "fp8/fp8"]`. The runner checks this list; it does not run quality evaluation itself. Calibration startup counts toward allocation time. CPU simulation only verifies wiring and cannot estimate the performance/accuracy benefit of FP8 weights.

Framework references: [vLLM FP8](https://docs.vllm.ai/en/latest/features/quantization/llm_compressor/fp8/), [SGLang quantization](https://docs.sglang.io/docs/advanced_features/quantization), [SGLang KV cache](https://github.com/sgl-project/sglang/blob/main/docs/docs/advanced_features/quantized_kv_cache.mdx), [TensorRT-LLM FP8](https://nvidia.github.io/TensorRT-LLM/1.1.0rc1/performance/performance-tuning-guide/fp8-quantization.html). Verify schemas against the pinned serving versions.


In [ ]:
# Independent Phase B axes. To force FP8 weights + FP8 cache, use ["fp8"] for both.
# Phase A always uses unquantized BF16 weights and the default cache (BF16).
PRECISION_CHOICES = {
    "vllm": {"weight_precision": ["bf16", "fp8"], "kv_cache_precision": ["bf16", "fp8"]},
    "sglang": {"weight_precision": ["bf16", "fp8"], "kv_cache_precision": ["bf16", "fp8"]},
    "trtllm": {"weight_precision": ["bf16", "fp8"], "kv_cache_precision": ["bf16", "fp8"]},
}
# TRT on-load weight quantization needs calibration. Adjust for your pinned version;
# prepare/cache its calibration dataset before launching the unattended campaign.
TRT_FP8_CALIBRATION = {
    "calib_batches": 32, "calib_batch_size": 1,
    "calib_max_seq_length": 512, "tokenizer_max_seq_length": 512,
}


In [ ]:
def make_runner():
    settings = {
        "wandb_group": CAMPAIGN,
        "frameworks": FRAMEWORKS, "real_frameworks": FRAMEWORKS,
        "run_real_gpu": RUN_REAL_GPU, "quick_demo": False,
        "phases": ["A", "B"], "concurrencies": CONCURRENCIES,
        "seeds": SEEDS, "sampler_names": ["tpe"], "pareto_samplers": ["tpe"],
        "phase_models": {"A": MODEL, "B": MODEL},
        "model_snapshots": {"A": MODEL_SNAPSHOT, "B": MODEL_SNAPSHOT},
        "phase_requests": {"A": REQUESTS, "B": REQUESTS},
        "search_trials": {p: {"1": TRIALS, "128": TRIALS} for p in ["A", "B"]},
        "base_workload": {
            "dataset_name": "random", "random_input_len": 512,
            "random_output_len": 128, "random_range_ratio": 1.0,
            "seed": 2026, "request_rate": "inf",
        },
        "validation_repeats": VALIDATION_REPEATS,
        "validation_requests": {"A": 1000, "B": 1000},
        "gpu_count": 1, "rest_seconds": 10,
        "real_preflight": {
            phase: {f: {**PREFLIGHT[f], "expanded_choices": EXPANDED_CHOICES[f]}
                    for f in FRAMEWORKS}
            for phase in ["A", "B"]
        },
    }
    return Phase1Runner(
        settings, BASE_SPACES, EXPANDED_CHOICES, EXTRA_SPACES, RUN_ROOT,
        precision_choices=PRECISION_CHOICES, trt_fp8_calibration=TRT_FP8_CALIBRATION,
    )

# Lazy construction keeps script imports free of run artifacts.
_runner = None

def get_runner():
    global _runner
    if _runner is None:
        _runner = make_runner()
    return _runner

def run_objective_group(phase, target, concurrencies, pruner_name="nop"):
    return get_runner().run_objective_group(phase, target, concurrencies, pruner_name)


## Independent experiment cells
Day 1: preflight, untuned baseline measurements, and recovery drills. Days 2–4: base (A) versus extended (B) searches on the same model, then median versus no pruning. Days 5–6: independent repeats and workload checks. Day 7: sustained-load confirmation. The queue below performs the search stage; use the validation cell afterwards. Measure trial duration during the pilot and adjust trial counts to fit the week. A finite completed queue exits successfully rather than repeating identical experiments indefinitely.

### A: single e2e (no pruning)

In [ ]:
A_single_e2e_studies = run_objective_group(
    "A", "e2e", [1], pruner_name="nop",
)
plot_group(A_single_e2e_studies)


### A: single ttft (no pruning)

In [ ]:
A_single_ttft_studies = run_objective_group(
    "A", "ttft", [1], pruner_name="nop",
)
plot_group(A_single_ttft_studies)


### A: loaded throughput (no pruning)

In [ ]:
A_loaded_throughput_studies = run_objective_group(
    "A", "throughput", [c for c in CONCURRENCIES if c > 1], pruner_name="nop",
)
plot_group(A_loaded_throughput_studies)


### A: loaded e2e (no pruning)

In [ ]:
A_loaded_e2e_studies = run_objective_group(
    "A", "e2e", [c for c in CONCURRENCIES if c > 1], pruner_name="nop",
)
plot_group(A_loaded_e2e_studies)


### B: single e2e (no pruning)

In [ ]:
B_single_e2e_studies = run_objective_group(
    "B", "e2e", [1], pruner_name="nop",
)
plot_group(B_single_e2e_studies)


### B: single ttft (no pruning)

In [ ]:
B_single_ttft_studies = run_objective_group(
    "B", "ttft", [1], pruner_name="nop",
)
plot_group(B_single_ttft_studies)


### B: loaded throughput (no pruning)

In [ ]:
B_loaded_throughput_studies = run_objective_group(
    "B", "throughput", [c for c in CONCURRENCIES if c > 1], pruner_name="nop",
)
plot_group(B_loaded_throughput_studies)


### B: loaded e2e (no pruning)

In [ ]:
B_loaded_e2e_studies = run_objective_group(
    "B", "e2e", [c for c in CONCURRENCIES if c > 1], pruner_name="nop",
)
plot_group(B_loaded_e2e_studies)


### B: loaded throughput with median pruning
The first ten trials establish pruning history. Decisions use measured request checkpoints, with at least three reference trials. The no-pruning comparison above uses the same workload and seeds.

In [ ]:
B_loaded_throughput_studies = run_objective_group(
    "B", "throughput", [c for c in CONCURRENCIES if c > 1],
    pruner_name="median",
)
plot_group(B_loaded_throughput_studies)


## Independent candidate validation
Run after search, within the remaining GPU budget. This uses the repository’s shortlist and independent validation workload/repeats; it is not an automatic sustained-load soak test. Keep the serving stack pinned and record any separate soak workload. Validation in this interactive cell does not use the batch watchdog.

In [ ]:
RUN_VALIDATION = False
if RUN_VALIDATION:
    from llm_optimizer.experiment.workflow import load_studies, write_json
    runner = get_runner()
    with runner.lock():
        studies = load_studies(runner.lab)
        validation = runner.lab.validate_candidates(runner.lab.shortlist(studies))
        write_json(runner.run_dir / "validation.json", runner.lab.artifact_value(validation))


## Unattended execution
Run `python notebooks/phase1_experiments.py` from the repository root. It runs each group in a fresh process group, terminates that entire group on timeout, and retries at most `MAX_RESTARTS` times. It records a persistent campaign deadline and skips completed tasks on restart. Nonzero final exit means one or more groups exhausted retries. Review failures instead of treating them as valid measurements.

Example systemd service (replace user, checkout and environment paths):
```ini
[Unit]
Description=Phase 1 GPU experiments
After=network-online.target
StartLimitIntervalSec=3600
StartLimitBurst=5
[Service]
Type=simple
User=ubuntu
WorkingDirectory=/workspace/llm-optimizer
ExecStart=/workspace/llm-optimizer/.venv/bin/python notebooks/phase1_experiments.py
Restart=on-failure
RestartSec=30
KillMode=control-group
TimeoutStopSec=30
Environment=PYTHONUNBUFFERED=1
Environment=CUDA_VISIBLE_DEVICES=0
[Install]
WantedBy=multi-user.target
```
Install as `/etc/systemd/system/phase1.service`, then run `sudo systemctl enable --now phase1`. Inspect `journalctl -u phase1 -f`. Keep clocks synchronized, durable disks attached, and artifacts backed up. The persisted 168-hour deadline survives service restarts. Transient failures and OOMs remain possible; the aim is preserved progress and bounded recovery.
